In [ ]:
import logging

from pathlib import Path

logging.basicConfig(level=logging.WARNING, force=True)
logger = logging.getLogger()

from aggregated_traces.utils.construct_ekg import (
    check_quantities,
    generate_networkx_di_graph,
    insert_DF_DP,
    insert_fractions,
    load_rdf_graph,
)
from aggregated_traces.utils.ekg_analysis import (
    compute_trace_probabilities,
    compute_number_of_merges_in_trace_graph,
)

In [ ]:
from rdflib import Graph, URIRef, Variable
from typing import List


def select_packing_units_with_quality_issue(graph: Graph, threshold: float) -> List[URIRef]:
    """
    Find packing unit(s) with average quality below a certain threshold.
    """

    query_lowest_quality = """
    PREFIX : <http://example.org/def/ekg/aggregated_traces/>
    SELECT DISTINCT
        ?packing_unit
        (sum(?device_quality)/count(?device_quality) as ?average_quality)
    WHERE {
        [] :bizStep "packing" ;
            :parentEntity ?packing_unit ;
            :device/:quality ?device_quality .

        ?packing_unit a :PackingUnit .
    }
    GROUP BY ?packing_unit
    """

    bindings = graph.query(query_lowest_quality).bindings

    return [
        b[Variable("packing_unit")]
        for b in bindings
        if b[Variable("average_quality")].toPython() < threshold
    ]


from pandas import DataFrame, Index


def get_entity_record_number(df: DataFrame, entity: str) -> int:
    try:
        return Index(df["entity_target"]).get_loc(entity) + 1
    except KeyError:
        return None

# Process event logs

## Compute trace probabilities

In [ ]:
scenario = "scenario_1"

base_path_logs = Path("logs")

# Define the root cause entity
root_cause_entity = URIRef("http://example.org/id/ekg/aggregated_traces/DB1")
quality_threshold = 0.8

# Define folder with event log files
event_log_files = base_path_logs.joinpath(scenario)

probability_dicts = {}
for event_data_file in event_log_files.glob("*.json"):
    print(event_data_file)

    # Parse data to RDF graph
    ekg_graph_rdf = load_rdf_graph(event_data_file)

    # Insert DF/DP relations
    ekg_graph_rdf = insert_DF_DP(ekg_graph_rdf)

    # Check incoming vs outgoing amount
    query_result_incoming_outgoing = check_quantities(ekg_graph_rdf)

    # Insert fractions on the relations
    ekg_graph_rdf = insert_fractions(ekg_graph_rdf)

    # Create NetworkX graph
    ekg_graph_nx = generate_networkx_di_graph(ekg_graph_rdf)

    # Find packing units with lowest (average) quality for which to find the root cause (entity)
    entities_source_backward = select_packing_units_with_quality_issue(graph=ekg_graph_rdf, threshold=quality_threshold)

    # Compute backward trace probabilities
    try:
        df_backward, edges_paths_backward = compute_trace_probabilities(
            rdf_trace_graph=ekg_graph_rdf,
            nx_trace_graph=ekg_graph_nx,
            source_entities=entities_source_backward,
            trace_backward=True,
        )
    except RuntimeError as e:
        logger.warning(e)
        continue

    for i in df_backward.index:
        df_backward.loc[i, "n_merges"] = compute_number_of_merges_in_trace_graph(
            trace_graph=ekg_graph_nx,
            source=df_backward.loc[i, "node_source"],
            target=df_backward.loc[i, "node_target"],
        )

    # Validate if probabilities add up to one
    if not all(
        df_backward.groupby(["entity_source", "product_model"])["probability"].sum()
        == 1.0
    ):
        raise Warning("Sum of probabilities by production step do not add up to one!")

    probability_dicts[event_data_file] = df_backward.to_dict(orient="records")

## Compute steps to find root cause (entity)

### Include all entities

In [6]:
results = []
for event_data_file, probability_dict in probability_dicts.items():
    df_trace = DataFrame(probability_dict)

    # Aggregate to get probabilities for target entities
    df_trace_grouped = df_trace.groupby(["entity_source", "entity_target"])[
        ["probability", "n_merges"]
    ].sum()
    df_trace_grouped.reset_index(inplace=True)

    entities_source = df_trace_grouped["entity_source"].unique()
    for entity_source in entities_source:
        df_selected = df_trace_grouped[
            df_trace_grouped["entity_source"] == entity_source
        ]

        # Shuffle/sample DataFrame for random order of inspection
        n_steps_random = get_entity_record_number(
            df_selected.sample(frac=1), root_cause_entity
        )

        # Sort values for informed order of inspection
        n_steps_sorted = get_entity_record_number(
            df_selected.sort_values("probability", ascending=False), root_cause_entity
        )

        results.append(
            {
                "file": event_data_file,
                "probabilities": df_selected.to_dict(orient="records"),
                "n_steps_random": n_steps_random,
                "n_steps_sorted": n_steps_sorted,
            }
        )

### Grouped by type (for example type of machine)

In [ ]:
group_key = "product_model"

results = []
for event_data_file, probability_dict in probability_dicts.items():
    df_trace = DataFrame(probability_dict)

    for group_value in df_trace[group_key].unique():
        # Aggregate to get probabilities for target entities
        df_trace_grouped = (
            df_trace[df_trace[group_key] == group_value]
            .groupby(["entity_source", "entity_target"])[["probability", "n_merges"]]
            .sum()
        )
        df_trace_grouped.reset_index(inplace=True)

        entities_source = df_trace_grouped["entity_source"].unique()
        for entity_source in entities_source:
            df_selected = df_trace_grouped[
                df_trace_grouped["entity_source"] == entity_source
            ]

            # Shuffle/sample DataFrame for random order of inspection
            n_steps_random = get_entity_record_number(
                df_selected.sample(frac=1), root_cause_entity
            )

            # Sort values for informed order of inspection
            n_steps_sorted = get_entity_record_number(
                df_selected.sort_values("probability", ascending=False),
                root_cause_entity,
            )

            results.append(
                {
                    "file": event_data_file,
                    group_key: group_value,
                    "probabilities": df_selected.to_dict(orient="records"),
                    "n_steps_random": n_steps_random,
                    "n_steps_sorted": n_steps_sorted,
                }
            )

In [ ]:
df = DataFrame(results)
df.describe()

# Statistics

In [ ]:
from scipy.stats import entropy

## Uniformity of probabilities

In [ ]:
for i in df.index:
    df_group = DataFrame(df.loc[i, "probabilities"])

    n_targets = df_group["entity_target"].nunique()
    probabilities = df_group.groupby(["entity_source", "entity_target"])[
        "probability"
    ].sum()
    probabilities = probabilities.astype(float)

    kl_divergence = entropy(pk=probabilities.values, qk=[1 / n_targets] * n_targets)

    df.loc[i, "kl_divergence"] = (
        kl_divergence  # closer to 0 is closer to uniform distribution
    )

### Number of merges

In [102]:
df["n_merges"] = df["probabilities"].apply(lambda x: sum(r["n_merges"] for r in x))

# Report results

In [103]:
df.to_csv(f"output/{scenario}.csv", index=False)

In [104]:
from matplotlib.pyplot import subplots

from pandas import concat, isna, read_csv, set_option

set_option("display.max_colwidth", None)

In [ ]:
scenarios = [1, 2, 3, 4]

df = DataFrame()
for i in scenarios:
    df = concat([df, read_csv(f"output/scenario_{i}.csv")])

In [ ]:
# If "n_steps_random" is NaN, the root cause entity does not occur in the trace
df[~isna(df["n_steps_random"])].describe()

In [ ]:
df.groupby("n_steps_random")[["kl_divergence", "n_merges"]].describe()
df.groupby("n_steps_sorted")[["kl_divergence", "n_merges"]].describe()

In [ ]:
x_column = "n_merges"
# x_column = "kl_divergence"
y_columns = {
    "n_steps_random": {
        "color": "lightblue",
        "marker": "s",
    },
    "n_steps_sorted": {
        "color": "green",
        "marker": ".",
    }
}

fig, ax = subplots(1,1)
for y_column, config in y_columns.items():
    df.plot.scatter(x=x_column, y=y_column, color=config["color"], marker=config["marker"], ax=ax, label=y_column)

ax.set_xlabel("KL divergence uniform distribution")
ax.set_ylabel("# machines inspected to find root cause")
ax.legend();